In [23]:
import pandas as pd
import numpy as np

In [24]:
# Set random seed for reproducibility
# Фіксуємо seed щоб результати були однакові при кожному запуску
np.random.seed(42)

# Total number of users who saw the ad
# Загальна кількість користувачів які побачили рекламу
n = 10000

# Create base dataframe with user attributes
# Створюємо базовий датафрейм з атрибутами користувачів
df = pd.DataFrame({
    'user_id': range(1, n + 1),

    # Acquisition channel / Канал залучення
    'channel': np.random.choice(
        ['Google', 'Facebook', 'TikTok', 'Organic'],
        n, p=[0.35, 0.30, 0.20, 0.15]
    ),

    # Country / Країна
    'country': np.random.choice(
        ['US', 'UK', 'DE', 'UA', 'Other'],
        n, p=[0.40, 0.20, 0.15, 0.10, 0.15]
    ),

    # Cohort month (when user first saw the ad)
    # Місяць когорти (коли користувач вперше побачив рекламу)
    'cohort_month': np.random.choice(
        ['2024-01', '2024-02', '2024-03', '2024-04'],
        n, p=[0.25, 0.25, 0.25, 0.25]
    ),

    # Step 1: Did the user click the ad? (45% click rate)
    # Крок 1: Чи клікнув користувач на рекламу? (45% click rate)
    'clicked': np.random.binomial(1, 0.45, n),
})

# Step 2: Install — only users who clicked can install (30% of clicks)
# Крок 2: Встановлення — тільки ті хто клікнув можуть встановити (30% від кліків)
df['installed'] = np.where(df['clicked'] == 1,
    np.random.binomial(1, 0.30, n), 0)

# Step 3: Onboarding — only installed users (65% complete onboarding)
# Крок 3: Онбординг — тільки ті хто встановив (65% завершують онбординг)
df['completed_onboarding'] = np.where(df['installed'] == 1,
    np.random.binomial(1, 0.65, n), 0)

# Step 4: Trial — only users who completed onboarding (50% start trial)
# Крок 4: Пробний період — тільки ті хто завершив онбординг (50% починають trial)
df['started_trial'] = np.where(df['completed_onboarding'] == 1,
    np.random.binomial(1, 0.50, n), 0)

# Step 5: Paid conversion — only trial users (35% convert to paid)
# Крок 5: Платна підписка — тільки ті хто почав trial (35% конвертуються)
df['converted_to_paid'] = np.where(df['started_trial'] == 1,
    np.random.binomial(1, 0.35, n), 0)

# Preview the data / Переглядаємо дані
print(df.head())
print(df.shape)

   user_id   channel country cohort_month  clicked  installed  \
0        1  Facebook      US      2024-03        1          0   
1        2   Organic      US      2024-01        0          0   
2        3    TikTok      US      2024-02        1          0   
3        4  Facebook      DE      2024-03        0          0   
4        5    Google      UK      2024-02        1          0   

   completed_onboarding  started_trial  converted_to_paid  
0                     0              0                  0  
1                     0              0                  0  
2                     0              0                  0  
3                     0              0                  0  
4                     0              0                  0  
(10000, 9)


In [26]:
# Funnel analysis — count users at each step
# Аналіз воронки — рахуємо користувачів на кожному кроці
funnel = pd.DataFrame({
    'step': [
        'Ad Impression',      # Побачили рекламу
        'Clicked',            # Клікнули
        'Installed',          # Встановили
        'Completed Onboarding', # Завершили онбординг
        'Started Trial',      # Почали trial
        'Converted to Paid'   # Стали платними
    ],
    'users': [
        len(df),
        df['clicked'].sum(),
        df['installed'].sum(),
        df['completed_onboarding'].sum(),
        df['started_trial'].sum(),
        df['converted_to_paid'].sum()
    ]
})

# Calculate conversion rate from previous step
# Конверсія відносно попереднього кроку
funnel['conversion_from_prev'] = (
    funnel['users'] / funnel['users'].shift(1) * 100
).round(1)

# Calculate conversion rate from the very first step
# Конверсія відносно самого початку воронки
funnel['conversion_from_start'] = (
    funnel['users'] / len(df) * 100
).round(1)

print(funnel.to_string(index=False))

                step  users  conversion_from_prev  conversion_from_start
       Ad Impression  10000                   NaN                  100.0
             Clicked   4512                  45.1                   45.1
           Installed   1353                  30.0                   13.5
Completed Onboarding    877                  64.8                    8.8
       Started Trial    453                  51.7                    4.5
   Converted to Paid    158                  34.9                    1.6


In [27]:
# Find the biggest drop-off step
# Знаходимо крок з найбільшим відсівом
funnel['drop_off'] = 100 - funnel['conversion_from_prev']

print("BIGGEST DROP-OFF")
print(funnel[['step', 'conversion_from_prev', 'drop_off']]
      .dropna()
      .sort_values('drop_off', ascending=False)
      .to_string(index=False))

BIGGEST DROP-OFF
                step  conversion_from_prev  drop_off
           Installed                  30.0      70.0
   Converted to Paid                  34.9      65.1
             Clicked                  45.1      54.9
       Started Trial                  51.7      48.3
Completed Onboarding                  64.8      35.2


In [28]:
# Funnel by acquisition channel
# Воронка по каналах залучення
channel_funnel = df.groupby('channel').agg(
    impressions=('user_id', 'count'),
    clicks=('clicked', 'sum'),
    installs=('installed', 'sum'),
    trials=('started_trial', 'sum'),
    paid=('converted_to_paid', 'sum')
).reset_index()

# Conversion rates per channel
# Конверсії по кожному каналу
channel_funnel['click_rate'] = (
    channel_funnel['clicks'] / channel_funnel['impressions'] * 100
).round(1)

channel_funnel['install_rate'] = (
    channel_funnel['installs'] / channel_funnel['clicks'] * 100
).round(1)

channel_funnel['trial_rate'] = (
    channel_funnel['trials'] / channel_funnel['installs'] * 100
).round(1)

channel_funnel['paid_rate'] = (
    channel_funnel['paid'] / channel_funnel['trials'] * 100
).round(1)

channel_funnel['overall_conversion'] = (
    channel_funnel['paid'] / channel_funnel['impressions'] * 100
).round(2)

print(channel_funnel[[
    'channel', 'impressions', 'click_rate',
    'install_rate', 'trial_rate', 'paid_rate', 'overall_conversion'
]].to_string(index=False))

 channel  impressions  click_rate  install_rate  trial_rate  paid_rate  overall_conversion
Facebook         3048        46.2          27.3        36.4       30.7                1.41
  Google         3555        44.4          32.8        32.6       40.2                1.91
 Organic         1448        44.5          28.9        31.7       27.1                1.10
  TikTok         1949        45.2          30.0        32.2       36.5                1.59


In [29]:
# Cohort analysis — overall conversion by cohort month
# Когортний аналіз — загальна конверсія по місяцях
cohort_analysis = df.groupby('cohort_month').agg(
    users=('user_id', 'count'),
    paid=('converted_to_paid', 'sum')
).reset_index()

cohort_analysis['conversion_rate'] = (
    cohort_analysis['paid'] / cohort_analysis['users'] * 100
).round(2)

print(cohort_analysis.to_string(index=False))

cohort_month  users  paid  conversion_rate
     2024-01   2465    47             1.91
     2024-02   2513    53             2.11
     2024-03   2575    30             1.17
     2024-04   2447    28             1.14


In [30]:
# Channel mix by cohort month
# Розподіл каналів по місяцях — чи змінився мікс?
channel_mix = df.groupby(['cohort_month', 'channel']).size().unstack()
channel_mix_pct = (channel_mix.div(channel_mix.sum(axis=1), axis=0) * 100).round(1)

print(channel_mix_pct.to_string())

channel       Facebook  Google  Organic  TikTok
cohort_month                                   
2024-01           30.6    35.3     13.6    20.4
2024-02           30.8    34.2     14.4    20.7
2024-03           30.5    35.4     15.3    18.8
2024-04           30.0    37.3     14.6    18.1


In [31]:
# Funnel analysis by country
# Аналіз воронки по країнах
country_funnel = df.groupby('country').agg(
    impressions=('user_id', 'count'),
    clicks=('clicked', 'sum'),
    installs=('installed', 'sum'),
    trials=('started_trial', 'sum'),
    paid=('converted_to_paid', 'sum')
).reset_index()

# Conversion rates by country
# Конверсії по країнах
country_funnel['install_rate'] = (
    country_funnel['installs'] / country_funnel['clicks'] * 100
).round(1)

country_funnel['trial_rate'] = (
    country_funnel['trials'] / country_funnel['installs'] * 100
).round(1)

country_funnel['paid_rate'] = (
    country_funnel['paid'] / country_funnel['trials'] * 100
).round(1)

country_funnel['overall_conversion'] = (
    country_funnel['paid'] / country_funnel['impressions'] * 100
).round(2)

print(country_funnel[[
    'country', 'impressions', 'install_rate',
    'trial_rate', 'paid_rate', 'overall_conversion'
]].to_string(index=False))

country  impressions  install_rate  trial_rate  paid_rate  overall_conversion
     DE         1483          30.4        34.2       42.6                1.96
  Other         1566          30.4        35.6       31.2                1.53
     UA         1006          28.8        39.5       41.2                2.09
     UK         2019          27.6        35.8       33.3                1.44
     US         3926          31.2        30.0       32.4                1.40


In [32]:
# Funnel by country AND channel
# Воронка по країнах і каналах одночасно
country_channel = df.groupby(['country', 'channel']).agg(
    impressions=('user_id', 'count'),
    paid=('converted_to_paid', 'sum')
).reset_index()

country_channel['overall_conversion'] = (
    country_channel['paid'] / country_channel['impressions'] * 100
).round(2)

# Pivot table — країни в рядках, канали в колонках
# Зручно читати як матрицю
pivot = country_channel.pivot_table(
    index='country',
    columns='channel',
    values='overall_conversion'
).round(2)

print(pivot.to_string())

channel  Facebook  Google  Organic  TikTok
country                                   
DE           1.20    2.81     1.28    2.00
Other        1.52    1.91     1.32    1.00
UA           1.60    2.49     2.88    1.55
UK           1.90    0.98     0.38    2.23
US           1.14    1.90     0.86    1.33


In [33]:
# Save all analysis results for Tableau
# Зберігаємо всі результати для Tableau

# Main funnel / Основна воронка
funnel.to_csv('funnel_overall.csv', index=False)

# Channel funnel / Воронка по каналах
channel_funnel.to_csv('funnel_by_channel.csv', index=False)

# Cohort analysis / Когортний аналіз
cohort_analysis.to_csv('funnel_by_cohort.csv', index=False)

# Channel mix by cohort / Канальний мікс по місяцях
channel_mix_pct.reset_index().to_csv('channel_mix.csv', index=False)

# Country funnel / Воронка по країнах
country_funnel.to_csv('funnel_by_country.csv', index=False)

# Country + Channel combination (long format for Tableau)
# Комбінація країна + канал (довгий формат для Tableau)
country_channel.to_csv('funnel_country_channel.csv', index=False)

# Raw data / Сирі дані
df.to_csv('subscription_funnel_data.csv', index=False)

print("All files saved! / Всі файли збережено!")

All files saved! / Всі файли збережено!


In [36]:
# Summary & Recommendations / Висновки та рекомендації

print("""
ENGLISH

Overall: 10,000 users → 158 paid (1.6% conversion)

Key Findings:
1. Install step has the highest drop-off (70%) — only 1,353 of 4,512 clicks result in an install.
   Review App Store page, ad creatives, and technical install issues.

2. Facebook has the lowest install rate (27.3%) — creatives attract irrelevant audience.

3. Google is the best channel (1.91% overall conversion, 40.2% paid rate) — increase budget.

4. Cohort conversion dropped 2x in March-April (from ~2% to ~1.1%).
   Channel mix stayed the same — likely a product change in March.

5. UA has the highest conversion (2.09%) despite the smallest audience — consider scaling.

6. US has the largest audience but lowest conversion (1.40%) — review targeting.

7. Best channel per country: DE → Google, UA → Organic, UK → TikTok, US → Google.
   No universal channel — ad strategy should be localized.

Priority actions:
1. Fix install funnel
2. Review Facebook creatives
3. Investigate March-April conversion drop
4. Scale Google globally, TikTok for UK
5. Localize strategy per country
""")


ENGLISH

Overall: 10,000 users → 158 paid (1.6% conversion)

Key Findings:
1. Install step has the highest drop-off (70%) — only 1,353 of 4,512 clicks result in an install.
   Review App Store page, ad creatives, and technical install issues.

2. Facebook has the lowest install rate (27.3%) — creatives attract irrelevant audience.

3. Google is the best channel (1.91% overall conversion, 40.2% paid rate) — increase budget.

4. Cohort conversion dropped 2x in March-April (from ~2% to ~1.1%).
   Channel mix stayed the same — likely a product change in March.

5. UA has the highest conversion (2.09%) despite the smallest audience — consider scaling.

6. US has the largest audience but lowest conversion (1.40%) — review targeting.

7. Best channel per country: DE → Google, UA → Organic, UK → TikTok, US → Google.
   No universal channel — ad strategy should be localized.

Priority actions:
1. Fix install funnel
2. Review Facebook creatives
3. Investigate March-April conversion drop
4. Scal

In [37]:
print("""
УКРАЇНСЬКА

Загалом: 10,000 користувачів → 158 платних (конверсія 1.6%)

Ключові висновки:
1. Найбільший відсів на етапі інсталу (70%) — лише 1,353 з 4,512 кліків завершились інсталом.
   Перевірити сторінку App Store, креативи, технічні проблеми з інсталяцією.

2. Facebook має найнижчий install rate (27.3%) — креативи залучають нерелевантну аудиторію.

3. Google — найкращий канал (конверсія 1.91%, paid rate 40.2%) — збільшити бюджет.

4. Конверсія когорт впала вдвічі у березні-квітні (з ~2% до ~1.1%).
   Канальний мікс не змінився — швидше за все зміна в продукті у березні.

5. UA має найвищу конверсію (2.09%) попри найменшу аудиторію — розглянути масштабування.

6. US має найбільшу аудиторію але найнижчу конверсію (1.40%) — переглянути таргетинг.

7. Найкращий канал по країнах: DE → Google, UA → Organic, UK → TikTok, US → Google.
   Немає універсального каналу — стратегія має бути локальною.

Пріоритетні дії:
1. Виправити воронку на етапі інсталу
2. Переглянути креативи Facebook
3. Дослідити падіння конверсії у березні-квітні
4. Масштабувати Google глобально, TikTok для UK
5. Локалізувати стратегію по країнах
""")


УКРАЇНСЬКА

Загалом: 10,000 користувачів → 158 платних (конверсія 1.6%)

Ключові висновки:
1. Найбільший відсів на етапі інсталу (70%) — лише 1,353 з 4,512 кліків завершились інсталом.
   Перевірити сторінку App Store, креативи, технічні проблеми з інсталяцією.

2. Facebook має найнижчий install rate (27.3%) — креативи залучають нерелевантну аудиторію.

3. Google — найкращий канал (конверсія 1.91%, paid rate 40.2%) — збільшити бюджет.

4. Конверсія когорт впала вдвічі у березні-квітні (з ~2% до ~1.1%).
   Канальний мікс не змінився — швидше за все зміна в продукті у березні.

5. UA має найвищу конверсію (2.09%) попри найменшу аудиторію — розглянути масштабування.

6. US має найбільшу аудиторію але найнижчу конверсію (1.40%) — переглянути таргетинг.

7. Найкращий канал по країнах: DE → Google, UA → Organic, UK → TikTok, US → Google.
   Немає універсального каналу — стратегія має бути локальною.

Пріоритетні дії:
1. Виправити воронку на етапі інсталу
2. Переглянути креативи Facebook
3. 